# PLOTTING FUNCTIONS

This module provides visualisation helpers for inspecting environmental sensor data collected across the case-study dwellings in the longitudinal IEQ study. The three functions plot time-series event dictionaries, 5-minute-resolution event columns from a DataFrame, and all variables in a DataFrame grouped by measurement type.

**Source file:** `data/code/plotting_functions.py`  
**Author:** Dr Cairan Van Rooyen (UCL)

## Import Packages

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## plot_grouped_variables

Iterates over a predefined list of measurement keywords and creates one publication-quality figure per keyword, overlaying all DataFrame columns whose name contains that keyword. Global `rcParams` are applied once at module load time. Each figure is saved as **PNG**, **PDF**, and **JPEG** at 300 DPI when `input_save=True`.

In [ ]:
# apply professional seaborn theme once at module load
sns.set_theme(
    style='ticks',
    palette='colorblind',
    font='Arial',
    rc={
        'figure.dpi':         120,
        'savefig.dpi':        300,
        'savefig.bbox':       'tight',
        'savefig.pad_inches': 0.05,
        'lines.linewidth':    0.9,
        'font.size':          9,
        'axes.titlesize':     11,
        'axes.titleweight':   'bold',
        'axes.labelsize':     9,
        'xtick.labelsize':    8,
        'ytick.labelsize':    8,
        'legend.fontsize':    8,
    }
)

# per-variable y-axis labels and figure titles
# mathtext notation ($...$) is used for sub/superscripts (arial lacks unicode glyph support)
_Y_LABELS = {
    'temperature': 'Temperature (\u00b0C)',
    'humidity':    'Relative Humidity (%)',
    'co2':         'CO$_2$ (ppm)',
    '1um':         'PM$_1$ (\u03bcg m$^{-3}$)',
    '25um':        'PM$_{2.5}$ (\u03bcg m$^{-3}$)',
    '100um':       'PM$_{10}$ (\u03bcg m$^{-3}$)',
    'no2':         'NO$_2$ (\u03bcg m$^{-3}$)',
    'co1':         'CO (\u03bcg m$^{-3}$)',
    'voc':         'tVOC (ppb)',
    'dba':         'Sound Level (dB(A))',
    'external':    'Measurement Value',
}

_TITLES = {
    'temperature': 'Temperature',
    'humidity':    'Relative Humidity',
    'co2':         'CO$_2$ Concentration',
    '1um':         'PM$_1$ Concentration',
    '25um':        'PM$_{2.5}$ Concentration',
    '100um':       'PM$_{10}$ Concentration',
    'no2':         'NO$_2$ Concentration',
    'co1':         'CO Concentration',
    'voc':         'tVOC Concentration',
    'dba':         'Indoor / Outdoor Sound Level',
    'external':    'External Measurements',
}


def plot_grouped_variables(input_dataframe, input_save=False, save_path='plots/', figsize=(10, 4), dpi=300):
    """
    Plot sensor variables from a DataFrame grouped by measurement type.

    For each measurement keyword (temperature, humidity, co2, etc.) a single
    figure is produced overlaying all DataFrame columns whose name contains
    that keyword. Figures are styled for scientific journal publication and
    optionally saved as PNG, PDF, and JPEG.

    Parameters
    ----------
    input_dataframe : pd.DataFrame
        DataFrame with a datetime index. Column names must follow the
        convention ``<room>_<variable>`` (e.g. ``kitchen_temperature``).
    input_save : bool, optional
        If True, save each figure as .png, .pdf, and .jpg. Default False.
    save_path : str, optional
        Output directory. Created automatically if absent. Default 'plots/'.
    figsize : tuple of float, optional
        Figure width and height in inches. Default (10, 4).
    dpi : int, optional
        Resolution for raster outputs (PNG, JPEG). Default 300.

    Returns
    -------
    None
    """
    if input_save:
        os.makedirs(save_path, exist_ok=True)  # create output directory if needed

    variables = [
        'temperature', 'humidity', 'co2',
        '1um', '25um', '100um',
        'no2', 'co1', 'voc', 'dba', 'external',
    ]

    for variable in variables:
        cols = [c for c in input_dataframe.columns if variable in c.lower()]  # find matching columns
        if not cols:
            continue  # skip silently if no columns match

        palette = sns.color_palette('colorblind', n_colors=len(cols))  # colorblind-safe colors
        fig, ax = plt.subplots(figsize=figsize)

        for col, color in zip(cols, palette):
            ax.plot(
                input_dataframe.index,
                input_dataframe[col],
                label=col.replace('_', ' ').title(),  # convert snake_case to title case
                color=color,
            )

        ax.set_title(_TITLES.get(variable, variable.capitalize()))
        ax.set_xlabel('Date')
        ax.set_ylabel(_Y_LABELS.get(variable, variable.capitalize()))
        ax.legend(loc='upper right', framealpha=0.9)
        fig.autofmt_xdate(rotation=30, ha='right')
        sns.despine(ax=ax)  # remove top and right spines
        fig.tight_layout()

        if input_save:
            base = os.path.join(save_path, variable.lower())
            fig.savefig(f'{base}.png', dpi=dpi)           # raster – png
            fig.savefig(f'{base}.pdf')                     # vector  – pdf
            fig.savefig(f'{base}.jpg', dpi=dpi,            # raster – jpeg
                        pil_kwargs={'quality': 95, 'subsampling': 0})

        plt.show()

## plot_event_timeseries

Creates one stacked subplot per event sensor in the event dictionary, plotting each as a binary open/closed step function over the monitoring period. Shading under each step aids readability. Saved as **PNG**, **PDF**, and **JPEG** at 300 DPI when `input_save=True`.

In [ ]:
def plot_event_timeseries(input_event_dict, input_start_date=None, input_end_date=None,
                          input_save=False, save_path='plots/', dpi=300):
    """
    Plot window and door event sensors as binary open/closed time series.

    Each event sensor is shown as a stacked subplot with a step function that
    toggles between 0 (closed) and 1 (open) at each recorded event timestamp.
    Sensors are assumed to start in the closed state.

    Parameters
    ----------
    input_event_dict : dict
        Dict of event DataFrames from ingest_ux90_001m_group_new. Each key is
        an event name and each value is a DataFrame with a 'date' column of
        event timestamps.
    input_start_date : str or datetime, optional
        Start of the plotting window. If None, uses the earliest event date.
    input_end_date : str or datetime, optional
        End of the plotting window. If None, uses the latest event date.
    input_save : bool, optional
        If True, save the figure as .png, .pdf, and .jpg. Default False.
    save_path : str, optional
        Output directory. Created automatically if absent. Default 'plots/'.
    dpi : int, optional
        Resolution for raster outputs (PNG, JPEG). Default 300.

    Returns
    -------
    None
    """
    # filter out event dataframes that are empty or missing a date column
    valid = {k: v for k, v in input_event_dict.items()
             if not v.empty and 'date' in v.columns}
    if not valid:
        print('no valid event data to plot')
        return

    # infer date range from data if not provided
    all_dates = pd.concat([df['date'] for df in valid.values()])
    start = pd.Timestamp(input_start_date) if input_start_date else all_dates.min()
    end   = pd.Timestamp(input_end_date)   if input_end_date   else all_dates.max()

    n       = len(valid)
    palette = sns.color_palette('colorblind', n_colors=n)
    fig, axes = plt.subplots(n, 1, figsize=(12, 1.6 * n), sharex=True)
    if n == 1:
        axes = [axes]  # ensure axes is always iterable

    for ax, (event_name, event_df), color in zip(axes, valid.items(), palette):
        # filter timestamps to monitoring period and sort
        timestamps = (
            pd.to_datetime(event_df['date'])
            .sort_values()
            .loc[lambda s: (s >= start) & (s <= end)]
            .reset_index(drop=True)
        )

        # build step function: starts closed (0), toggles at each event
        times  = [start] + list(timestamps) + [end]
        states = [0]
        for _ in timestamps:
            states.append(1 - states[-1])  # toggle open/closed
        states.append(states[-1])           # hold final state to end date

        ax.step(times, states, where='post', color=color, linewidth=1.0)
        ax.fill_between(times, states, step='post', alpha=0.2, color=color)  # shaded fill

        label = event_name.replace('_event', '').replace('_', ' ').title()
        ax.set_ylabel(label, fontsize=8)
        ax.set_ylim(-0.15, 1.15)
        ax.set_yticks([0, 1])
        ax.set_yticklabels(['Closed', 'Open'], fontsize=7)
        sns.despine(ax=ax)

    axes[-1].set_xlabel('Date')
    fig.suptitle('Window and Door Events', fontsize=11, fontweight='bold')
    fig.autofmt_xdate(rotation=30, ha='right')
    fig.tight_layout()

    if input_save:
        os.makedirs(save_path, exist_ok=True)
        base = os.path.join(save_path, 'events')
        fig.savefig(f'{base}.png', dpi=dpi)           # raster – png
        fig.savefig(f'{base}.pdf')                     # vector  – pdf
        fig.savefig(f'{base}.jpg', dpi=dpi,            # raster – jpeg
                    pil_kwargs={'quality': 95, 'subsampling': 0})

    plt.show()